# How to visualize your Dot Product Data



First we have to declare our **libraries**.

In [ ]:
import pandas as pd                 
import matplotlib.pyplot as plt
import numpy as np

Next, we define the **path**.

In [ ]:
PATH = '~/Library/Mobile Documents/com~apple~CloudDocs/Desktop/'
FILENAME_POS9_Angle = 'Time and theta_POS9.csv'
data = pd.read_csv(PATH + FILENAME_POS9_Angle, sep=',')

Then, we have to **extrapolate the data** from the file. If you get an error here, make sure the header in the data file matches the name you have in your code.

In [ ]:
Time = data['Time(min)'] - data['Time(min)'].min() 
Angle = data['Theta(Degrees)']
data['Time_normalized'] = Time

Next, we declare our **bins** that will hold our data. 

In [ ]:
bin_size = 3  #time step

max_time = Time.max()

time_bins = np.arange(0, max_time + bin_size, bin_size) 

time_centers = []
anti_para_pcts = []
ortho_pcts = []
para_pcts = []
rando_pcts = []

Now we can **break our results into specific slots**. We want to see how many are parallel, antiparallel, orthogonal or just random.

We put all of our incoming data into the **data bin**.

Then we find the number of results that fall under each category and create that into a **%**.

In [ ]:
for i in range(len(time_bins) - 1):
    bin_mask = (Time >= time_bins[i]) & (Time < time_bins[i + 1])
    bin_data = data[bin_mask]
    
    if len(bin_data) > 0:  
        anti_parallel_count = len(bin_data[bin_data['Theta(Degrees)'] > 165])
        orthogonal_count = len(bin_data[(bin_data['Theta(Degrees)'] > 82.5) & (bin_data['Theta(Degrees)'] < 97.5)])
        parallel_count = len(bin_data[(bin_data['Theta(Degrees)'] > 0) & (bin_data['Theta(Degrees)'] < 15)])
        rando_count = len(bin_data) - anti_parallel_count - orthogonal_count - parallel_count
        total_count = len(bin_data)
        time_centers.append(time_bins[i]) 
        anti_para_pcts.append((anti_parallel_count / total_count) * 100)
        ortho_pcts.append((orthogonal_count / total_count) * 100)
        para_pcts.append((parallel_count / total_count) * 100)
        rando_pcts.append((rando_count / total_count) * 100)


So the first way we can 'visualize' this is by printing it our **minute by minute** and see what happens.

In [ ]:
print("Time-based percentages (3-minute intervals starting from 0):")
for i, time_start in enumerate(time_centers):
    print(f"Time {time_start:.1f} min: Anti-parallel: {anti_para_pcts[i]:.1f}%, "
          f"Orthogonal: {ortho_pcts[i]:.1f}%, Parallel: {para_pcts[i]:.1f}%, Random: {rando_pcts[i]:.1f}%")

This is quite messy though, so lets **plot** it.

In [ ]:
plt.figure(figsize=(25, 5))
plt.plot(time_centers, anti_para_pcts, 'o-', label='Anti-parallel (>165°)', linewidth=1)
plt.plot(time_centers, ortho_pcts, 's-', label='Orthogonal (82.5-97.5°)', linewidth=1)
plt.plot(time_centers, para_pcts, '^-', label='Parallel (0-15°)', linewidth=1)
#plt.plot(time_centers, rando_pcts, 'd-', label='Random', linewidth=1)
plt.xlabel('Time (minutes from start)')
plt.ylabel('Percentage (%)')
plt.title('Angle Category Percentages Over Time (3-minute intervals)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Much better!

We can also determine how some of our results are **outliers**. We will use the same method as the image analysis plots by first **normalizing** the data. 

In [ ]:
#normalization of data to standard deviations 
def rolling_zscore(series, window):
    if isinstance(series, list):
        series = pd.Series(series)
    rolling_mean = series.rolling(window=window, center=True).mean()
    rolling_std = series.rolling(window=window, center=True).std()
    std = (series - rolling_mean) / rolling_std
    if (std > 2.5).any():
        print("Outlier values: ", series[std>2.5])
    return std
window_frames = int(300 / 3.0) 
norm_orthodata = rolling_zscore(ortho_pcts, window_frames)
norm_paralleldata = rolling_zscore(para_pcts, window_frames)
norm_antiparadata = rolling_zscore(anti_para_pcts, window_frames)

valid_indices = ~(norm_orthodata.isna() | norm_paralleldata.isna() | norm_antiparadata.isna())

Next we can try to smooth the data with a **fourier transform** although in practice, this was not always successful.

Change the threshold_percent value to smooth. The larger it is the smoother it will be. For raw results, keep set at 0.0.

In [ ]:
#ft of data 
def fft_denoise(signal, threshold_percent=.05):
    fft_vals = np.fft.fft(signal)
    power = np.abs(fft_vals) ** 2
    max_power = np.max(power)
    threshold = threshold_percent * max_power
    fft_vals_clean = fft_vals.copy()
    fft_vals_clean[power < threshold] = 0
    return np.fft.ifft(fft_vals_clean).real
clean_ortho = fft_denoise(norm_orthodata.dropna(), threshold_percent=0.0)
clean_para = fft_denoise(norm_paralleldata.dropna(), threshold_percent=0.0)
clean_antipara = fft_denoise(norm_antiparadata.dropna(), threshold_percent=0.0)

#bin count changes, must update or error will throw
time_centers_aligned = np.array(time_centers)[valid_indices]

Lets **plot** this new, normalzied, smoothed data. 

In [ ]:
plt.figure(figsize=(25,5))
plt.plot(time_centers_aligned, clean_ortho, label='Orthogonal (82.5-97.5°)', linewidth=1, color='orange')
plt.plot(time_centers_aligned, clean_para, label='Parallel (0-15°)', linewidth=1, color='green')
plt.plot(time_centers_aligned, clean_antipara, label='Anti-parallel (>165°)', linewidth=1, color='gray')

### Some Helpful Notes:

This data is super noisy! So it might be helpful to plot each of the behaviors seperatley. 

It might also be helpful to plot your Mechanical and or Mitotic activity relative to these directional behaviors. You can see what some of this might look like in one of my presentation slides. 

Use your discretion or intution to see plot what you would like. There are plenty of combinations you can explore!

# End of Code!